# TPC-H Collector Demo

# Missing
* connection data beyond loading
* join to connection data
* join to monitoring_single
* connection and configuration also in monitoring and connection df

## Naming
* There is an experiment having a code, say `1775855486`.
* The experiment inspects a SUT, say `PostgreSQL-A`. This is called a `configuration`.
* The experiment is run several times, say twice. The indicator of the run is called `experiment_run`.
* Each run can have several phases as a sequence. The number of the phase is called `client`. The state of the configuration in a phase is called a `connection`.
* Each client can have several `pods`, that are run in parallel. A pod represents a driver.
* Performance metrics are collected per driver pod.  
    The naming of an instance is `<sut>-<experiment_run>-<client>-<pod>`. It is unique per experiment.
* Monitoring metrics are collected per phase. They are automatically aggregated across parallel pods.  
    The naming of an instance is `<sut>-<experiment_run>-<client>`. It is unique per experiment.

## Aggregation
* Aggregation is complicated. Some metrics are aggregated via count, sum, max or average. Others cannot be aggregated sensibly, like experiment code or latency percentiles.
* There are helper functions to aggregated pods that are certainly run in parallel.  
  So `<sut>-<experiment_run>-<client>-<pod>` are aggregated to `<sut>-<experiment_run>-<client>`.
* An exception is multi-tenancy.

## Class `collector`
* constructor `collectors.benchbase(path, codes)`
* evaluator object `evaluate = collect.get_evaluator()`
* dataframe of connection infos `collect.get_connections()`
  * index is name of client
* **monitoring metrics**
  * dataframe of available metrics `collect.df_metrics`
    * index is key of metric
  * dataframe of monitored components `collect.get_monitored_phases()`
    * index is key of component
  * dataframe of aggregated metrics per connection `collect.get_monitoring_single_all()`
    * index is name of client
    * metrics aggregated per code, experiment_run and client
  * dataframe of aggregated metrics per connection and across tenants `collect.get_monitoring_all()`
    * index just enumerates
    * metrics aggregated per code, experiment_run and client and across tenants
  * dataframe of time series for a metric of a connection in an experiment `collect.get_monitoring_timeseries_single(code, metric)`
    * index is name of connection
    * unstacked (wide format)
  * dataframe of time series for a metric in all experiments `collect.get_monitoring_timeseries_all(metric)`
    * index just enumerates
    * stacked (long format)
* **performance metrics**
  * dataframe of loading metrics `collect.get_loading_time_max_all()`
    * index is name of client
  * dataframe of performance aggregated per parallel client `collect.get_performance_all()`
    * index just enumerates
    * performance aggregated per code, experiment_run and client
  * dataframe of performance for one experiment `collect.get_performance_single()`
    * performance of each single client (driver)
    * index is name of client pod
  * dataframe of performance for all experiments `collect.get_performance_all_single()`
    * performance of each single client (driver)
    * index is name of client pod


[1] [Benchmarking Multi-Tenant Architectures in PostgreSQL](https://doi.org/10.48786/edbt.2026.46)
> Erdelt, P.K., and Rabl T. (2026)
> In: Proceedings 29th International Conference on Extending Database Technology, EDBT 2026
> OpenProceedings.org
> https://doi.org/10.48786/edbt.2026.46


In [1]:
import pandas as pd
pd.set_option("display.max_rows", None)
pd.set_option('display.max_colwidth', None)
pd.options.display.max_columns = None
pd.options.display.max_rows = None
pd.options.display.float_format = '{:.2f}'.format

from bexhoma import collectors

%matplotlib inline

# Functions for Nice Plots

In [2]:
from notebookutils import *

# Collect Results

In [3]:
#path = r"D:\data\benchmarks"
path = r"/data/benchmarks"
filename_prefix = "demo_"
b_plot_save = False
b_skip_plots = True

In [4]:
codes = [
    "1776418956",
    "1776420958"
]

codes

['1776418956', '1776420958']

In [5]:
collect = collectors.default(path, codes)

# Get all Metrics Metadata

## Metrics Names and Types

Metrics that are derived from monitoring

In [6]:
collect.df_metrics

,title,active,type,metric
total_cpu_memory,Memory Usage [MiB],True,cluster,gauge
total_cpu_memory_cached,Memory Usage Cached [MiB],True,cluster,gauge
total_cpu_util,CPU Utilization,True,cluster,gauge
total_cpu_throttled,CPU Throttle,True,cluster,gauge
total_cpu_throttled_s,CPU Throttled Time [s],True,cluster,counter
total_cpu_util_others,CPU Utilization Others,False,cluster,gauge
total_cpu_util_s,CPU Utilization Time [s],True,cluster,counter
total_cpu_util_user_s,CPU User Time [s],True,cluster,counter
total_cpu_util_sys_s,CPU System Time [s],True,cluster,counter
total_cpu_util_others_s,CPU Utilization Time Others [s],False,cluster,counter


## Names of Monitored Components

Names of components and their phases

In [7]:
collect.get_monitored_components()

,description
loading,Loading phase: SUT deployment
datagenerator,Loading phase: component data generator
loader,Loading phase: component loader
stream,Execution phase: SUT deployment
benchmarker,Execution phase: component benchmarker


# Get Connection Infos

List of states of SUTs, containing loading infos.

In [8]:
df=collect.get_connections()

In [9]:
df.T

,1776418956-PostgreSQL-BHT-8-1-1,1776418956-PostgreSQL-BHT-8-1-2,1776418956-PostgreSQL-BHT-8-2-1,1776418956-PostgreSQL-BHT-8-2-2,1776420958-PostgreSQL-BHT-8-1-1,1776420958-PostgreSQL-BHT-8-1-2,1776420958-PostgreSQL-BHT-8-2-1,1776420958-PostgreSQL-BHT-8-2-2
code,1776418956,1776418956,1776418956,1776418956,1776420958,1776420958,1776420958,1776420958
connection,PostgreSQL-BHT-8-1-1-1,PostgreSQL-BHT-8-1-2-2,PostgreSQL-BHT-8-2-1-1,PostgreSQL-BHT-8-2-2-2,PostgreSQL-BHT-8-1-1-1,PostgreSQL-BHT-8-1-2-2,PostgreSQL-BHT-8-2-1-1,PostgreSQL-BHT-8-2-2-2
configuration,,,,,,,,
phase,1776418956-PostgreSQL-BHT-8-1-1,1776418956-PostgreSQL-BHT-8-1-2,1776418956-PostgreSQL-BHT-8-2-1,1776418956-PostgreSQL-BHT-8-2-2,1776420958-PostgreSQL-BHT-8-1-1,1776420958-PostgreSQL-BHT-8-1-2,1776420958-PostgreSQL-BHT-8-2-1,1776420958-PostgreSQL-BHT-8-2-2
experiment_run,1,1,2,2,1,1,2,2
client,1,2,1,2,1,2,1,2
dockerimage,postgres:18.3,postgres:18.3,postgres:18.3,postgres:18.3,postgres:18.3,postgres:18.3,postgres:18.3,postgres:18.3
time_load,455.00,455.00,455.00,455.00,1256.00,1256.00,1256.00,1256.00
time_ingest,141.00,141.00,141.00,141.00,350.00,350.00,350.00,350.00
time_check,291.00,291.00,291.00,291.00,843.00,843.00,843.00,843.00


In [10]:
df[['phase', 'code', 'connection', 'configuration', 'experiment_run', 'client', 'type_tenants', 'num_tenants', 'vol_tenants']]

,phase,code,connection,configuration,experiment_run,client,type_tenants,num_tenants,vol_tenants
1776418956-PostgreSQL-BHT-8-1-1,1776418956-PostgreSQL-BHT-8-1-1,1776418956,PostgreSQL-BHT-8-1-1-1,,1,1,,0,False
1776418956-PostgreSQL-BHT-8-1-2,1776418956-PostgreSQL-BHT-8-1-2,1776418956,PostgreSQL-BHT-8-1-2-2,,1,2,,0,False
1776418956-PostgreSQL-BHT-8-2-1,1776418956-PostgreSQL-BHT-8-2-1,1776418956,PostgreSQL-BHT-8-2-1-1,,2,1,,0,False
1776418956-PostgreSQL-BHT-8-2-2,1776418956-PostgreSQL-BHT-8-2-2,1776418956,PostgreSQL-BHT-8-2-2-2,,2,2,,0,False
1776420958-PostgreSQL-BHT-8-1-1,1776420958-PostgreSQL-BHT-8-1-1,1776420958,PostgreSQL-BHT-8-1-1-1,,1,1,,0,False
1776420958-PostgreSQL-BHT-8-1-2,1776420958-PostgreSQL-BHT-8-1-2,1776420958,PostgreSQL-BHT-8-1-2-2,,1,2,,0,False
1776420958-PostgreSQL-BHT-8-2-1,1776420958-PostgreSQL-BHT-8-2-1,1776420958,PostgreSQL-BHT-8-2-1-1,,2,1,,0,False
1776420958-PostgreSQL-BHT-8-2-2,1776420958-PostgreSQL-BHT-8-2-2,1776420958,PostgreSQL-BHT-8-2-2-2,,2,2,,0,False


In [11]:
collectors.get_non_constant(df).T

,1776418956-PostgreSQL-BHT-8-1-1,1776418956-PostgreSQL-BHT-8-1-2,1776418956-PostgreSQL-BHT-8-2-1,1776418956-PostgreSQL-BHT-8-2-2,1776420958-PostgreSQL-BHT-8-1-1,1776420958-PostgreSQL-BHT-8-1-2,1776420958-PostgreSQL-BHT-8-2-1,1776420958-PostgreSQL-BHT-8-2-2
code,1776418956,1776418956,1776418956,1776418956,1776420958,1776420958,1776420958,1776420958
connection,PostgreSQL-BHT-8-1-1-1,PostgreSQL-BHT-8-1-2-2,PostgreSQL-BHT-8-2-1-1,PostgreSQL-BHT-8-2-2-2,PostgreSQL-BHT-8-1-1-1,PostgreSQL-BHT-8-1-2-2,PostgreSQL-BHT-8-2-1-1,PostgreSQL-BHT-8-2-2-2
phase,1776418956-PostgreSQL-BHT-8-1-1,1776418956-PostgreSQL-BHT-8-1-2,1776418956-PostgreSQL-BHT-8-2-1,1776418956-PostgreSQL-BHT-8-2-2,1776420958-PostgreSQL-BHT-8-1-1,1776420958-PostgreSQL-BHT-8-1-2,1776420958-PostgreSQL-BHT-8-2-1,1776420958-PostgreSQL-BHT-8-2-2
experiment_run,1,1,2,2,1,1,2,2
client,1,2,1,2,1,2,1,2
time_load,455.00,455.00,455.00,455.00,1256.00,1256.00,1256.00,1256.00
time_ingest,141.00,141.00,141.00,141.00,350.00,350.00,350.00,350.00
time_check,291.00,291.00,291.00,291.00,843.00,843.00,843.00,843.00
pods,1,2,1,2,1,2,1,2
datadisk,8187,8187,8187,8187,16350,16350,16350,16350


# Monitoring Aggregated per Phase

In [12]:
df = collect.get_monitoring_aggregated_per_phase("stream")
df.T

,1776418956-PostgreSQL-BHT-8-1-1,1776418956-PostgreSQL-BHT-8-1-2,1776418956-PostgreSQL-BHT-8-2-1,1776418956-PostgreSQL-BHT-8-2-2,1776420958-PostgreSQL-BHT-8-1-1,1776420958-PostgreSQL-BHT-8-1-2,1776420958-PostgreSQL-BHT-8-2-1,1776420958-PostgreSQL-BHT-8-2-2
Memory Usage [MiB],10919.47,12131.24,10392.28,11955.10,17504.07,19192.60,14054.22,16993.59
Memory Usage Cached [MiB],15835.40,17047.17,15288.83,16789.84,27302.43,28990.96,22555.50,26276.44
CPU Utilization,1.66,3.29,0.96,3.39,2.91,5.70,1.60,5.64
CPU Throttle,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
CPU Throttled Time [s],0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
CPU Utilization Time [s],180.14,320.88,998.63,395.04,381.59,869.22,2263.06,877.07
CPU User Time [s],156.36,281.35,826.53,344.83,331.65,767.27,1928.57,768.57
CPU System Time [s],23.78,39.53,172.10,50.21,49.94,101.95,334.50,108.51
Network Rx Total [MiB],1.72,7.30,3307.36,1.87,1.91,3.74,6644.39,3.68
Network Tx Total [MiB],0.20,1320.29,21.81,3.35,1.96,3.78,39.78,3.66


### With Metadata

In [13]:
df = collect.add_metadata(df)
df.T

combine on index and column 'phase'


phase,1776418956-PostgreSQL-BHT-8-1-1,1776418956-PostgreSQL-BHT-8-1-2,1776418956-PostgreSQL-BHT-8-2-1,1776418956-PostgreSQL-BHT-8-2-2,1776420958-PostgreSQL-BHT-8-1-1,1776420958-PostgreSQL-BHT-8-1-2,1776420958-PostgreSQL-BHT-8-2-1,1776420958-PostgreSQL-BHT-8-2-2
Memory Usage [MiB],10919.47,12131.24,10392.28,11955.10,17504.07,19192.60,14054.22,16993.59
Memory Usage Cached [MiB],15835.40,17047.17,15288.83,16789.84,27302.43,28990.96,22555.50,26276.44
CPU Utilization,1.66,3.29,0.96,3.39,2.91,5.70,1.60,5.64
CPU Throttle,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
CPU Throttled Time [s],0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
CPU Utilization Time [s],180.14,320.88,998.63,395.04,381.59,869.22,2263.06,877.07
CPU User Time [s],156.36,281.35,826.53,344.83,331.65,767.27,1928.57,768.57
CPU System Time [s],23.78,39.53,172.10,50.21,49.94,101.95,334.50,108.51
Network Rx Total [MiB],1.72,7.30,3307.36,1.87,1.91,3.74,6644.39,3.68
Network Tx Total [MiB],0.20,1320.29,21.81,3.35,1.96,3.78,39.78,3.66


In [14]:
df[['phase', 'code', 'configuration', 'experiment_run', 'client', 'type_tenants', 'num_tenants', 'vol_tenants']].T

phase,1776418956-PostgreSQL-BHT-8-1-1,1776418956-PostgreSQL-BHT-8-1-2,1776418956-PostgreSQL-BHT-8-2-1,1776418956-PostgreSQL-BHT-8-2-2,1776420958-PostgreSQL-BHT-8-1-1,1776420958-PostgreSQL-BHT-8-1-2,1776420958-PostgreSQL-BHT-8-2-1,1776420958-PostgreSQL-BHT-8-2-2
phase,1776418956-PostgreSQL-BHT-8-1-1,1776418956-PostgreSQL-BHT-8-1-2,1776418956-PostgreSQL-BHT-8-2-1,1776418956-PostgreSQL-BHT-8-2-2,1776420958-PostgreSQL-BHT-8-1-1,1776420958-PostgreSQL-BHT-8-1-2,1776420958-PostgreSQL-BHT-8-2-1,1776420958-PostgreSQL-BHT-8-2-2
code,1776418956,1776418956,1776418956,1776418956,1776420958,1776420958,1776420958,1776420958
configuration,,,,,,,,
experiment_run,1,1,2,2,1,1,2,2
client,1,2,1,2,1,2,1,2
type_tenants,,,,,,,,
num_tenants,0,0,0,0,0,0,0,0
vol_tenants,False,False,False,False,False,False,False,False


# Time Series of a Single Metric for a Single Experiment

In [15]:
metric = 'total_cpu_memory'
code = codes[0]
df = collect.get_monitoring_timeseries_per_phase(code, metric=metric, component='stream')
df.T

,1776418956-PostgreSQL-BHT-8-1-1,1776418956-PostgreSQL-BHT-8-1-2,1776418956-PostgreSQL-BHT-8-2-1,1776418956-PostgreSQL-BHT-8-2-2
0,9770.97,10315.59,10315.64,10300.51
1,9770.97,10315.59,10315.64,10300.51
2,9770.97,10315.59,10315.64,10300.51
3,9770.97,10315.59,10315.64,10300.51
4,9770.97,10315.59,10315.64,10300.51
5,9838.88,10315.59,10315.64,10300.51
6,9838.88,10315.59,10315.64,10300.51
7,9838.88,10315.59,10315.64,10300.51
8,9838.88,10315.59,10315.64,10300.51
9,9838.88,10315.59,10315.64,10300.51


### With Metadata

In [16]:
df = collect.add_metadata(df)
df.T

combine on index and column 'phase'


phase,1776418956-PostgreSQL-BHT-8-1-1,1776418956-PostgreSQL-BHT-8-1-2,1776418956-PostgreSQL-BHT-8-2-1,1776418956-PostgreSQL-BHT-8-2-2
0,9770.97,10315.59,10315.64,10300.51
1,9770.97,10315.59,10315.64,10300.51
2,9770.97,10315.59,10315.64,10300.51
3,9770.97,10315.59,10315.64,10300.51
4,9770.97,10315.59,10315.64,10300.51
5,9838.88,10315.59,10315.64,10300.51
6,9838.88,10315.59,10315.64,10300.51
7,9838.88,10315.59,10315.64,10300.51
8,9838.88,10315.59,10315.64,10300.51
9,9838.88,10315.59,10315.64,10300.51


In [17]:
df[['phase', 'code', 'configuration', 'experiment_run', 'client', 'type_tenants', 'num_tenants', 'vol_tenants']].head()

,phase,code,configuration,experiment_run,client,type_tenants,num_tenants,vol_tenants
phase,,,,,,,,
1776418956-PostgreSQL-BHT-8-1-1,1776418956-PostgreSQL-BHT-8-1-1,1776418956,,1,1,,0,False
1776418956-PostgreSQL-BHT-8-1-2,1776418956-PostgreSQL-BHT-8-1-2,1776418956,,1,2,,0,False
1776418956-PostgreSQL-BHT-8-2-1,1776418956-PostgreSQL-BHT-8-2-1,1776418956,,2,1,,0,False
1776418956-PostgreSQL-BHT-8-2-2,1776418956-PostgreSQL-BHT-8-2-2,1776418956,,2,2,,0,False


# Time Series of a Single Metric for All Experiments

In [18]:
metric = 'total_cpu_memory'
code = codes[0]
df = collect.get_monitoring_timeseries_all(metric=metric)
df.head()

,timestamp,code,phase,experiment_run,client,type_tenants,vol_tenants,num_tenants,metric,component,value
0,0,1776418956,1776418956-PostgreSQL-BHT-8-1-1,1,1,,False,0,total_cpu_memory,stream,9770.97
1,0,1776418956,1776418956-PostgreSQL-BHT-8-1-2,1,2,,False,0,total_cpu_memory,stream,10315.59
2,0,1776418956,1776418956-PostgreSQL-BHT-8-2-1,2,1,,False,0,total_cpu_memory,stream,10315.64
3,0,1776418956,1776418956-PostgreSQL-BHT-8-2-2,2,2,,False,0,total_cpu_memory,stream,10300.51
4,0,1776420958,1776420958-PostgreSQL-BHT-8-1-1,1,1,,False,0,total_cpu_memory,stream,13877.75


In [19]:
df[['timestamp', 'phase', 'value']].head(8)

,timestamp,phase,value
0,0,1776418956-PostgreSQL-BHT-8-1-1,9770.97
1,0,1776418956-PostgreSQL-BHT-8-1-2,10315.59
2,0,1776418956-PostgreSQL-BHT-8-2-1,10315.64
3,0,1776418956-PostgreSQL-BHT-8-2-2,10300.51
4,0,1776420958-PostgreSQL-BHT-8-1-1,13877.75
5,0,1776420958-PostgreSQL-BHT-8-1-2,14874.78
6,0,1776420958-PostgreSQL-BHT-8-2-1,14904.29
7,0,1776420958-PostgreSQL-BHT-8-2-2,14743.28


In [20]:
df=collect.add_metadata(df)
df = df.sort_values(['timestamp', 'phase'])
df[['timestamp', 'phase', 'value']].head(8)

combine on columns phase


,timestamp,phase,value
1776418956-PostgreSQL-BHT-8-1-1,0,1776418956-PostgreSQL-BHT-8-1-1,9770.97
1776418956-PostgreSQL-BHT-8-1-2,0,1776418956-PostgreSQL-BHT-8-1-2,10315.59
1776418956-PostgreSQL-BHT-8-2-1,0,1776418956-PostgreSQL-BHT-8-2-1,10315.64
1776418956-PostgreSQL-BHT-8-2-2,0,1776418956-PostgreSQL-BHT-8-2-2,10300.51
1776420958-PostgreSQL-BHT-8-1-1,0,1776420958-PostgreSQL-BHT-8-1-1,13877.75
1776420958-PostgreSQL-BHT-8-1-2,0,1776420958-PostgreSQL-BHT-8-1-2,14874.78
1776420958-PostgreSQL-BHT-8-2-1,0,1776420958-PostgreSQL-BHT-8-2-1,14904.29
1776420958-PostgreSQL-BHT-8-2-2,0,1776420958-PostgreSQL-BHT-8-2-2,14743.28


In [21]:
df[['phase', 'code', 'configuration', 'experiment_run', 'client', 'type_tenants', 'num_tenants', 'vol_tenants']].head()

,phase,code,configuration,experiment_run,client,type_tenants,num_tenants,vol_tenants
1776418956-PostgreSQL-BHT-8-1-1,1776418956-PostgreSQL-BHT-8-1-1,1776418956,,1,1,,0,False
1776418956-PostgreSQL-BHT-8-1-2,1776418956-PostgreSQL-BHT-8-1-2,1776418956,,1,2,,0,False
1776418956-PostgreSQL-BHT-8-2-1,1776418956-PostgreSQL-BHT-8-2-1,1776418956,,2,1,,0,False
1776418956-PostgreSQL-BHT-8-2-2,1776418956-PostgreSQL-BHT-8-2-2,1776418956,,2,2,,0,False
1776420958-PostgreSQL-BHT-8-1-1,1776420958-PostgreSQL-BHT-8-1-1,1776420958,,1,1,,0,False


## Custom Aggregation and Scale

In [22]:
metric = 'total_cpu_util'

df_performance_series = collect.get_monitoring_timeseries_all(metric)

df_agg = (
    df_performance_series.groupby(["code", "experiment_run", "client", "type_tenants", "num_tenants"])["value"]
      .max()
      .reset_index()
)
#plot_bars(df_agg, y='value', title='Max CPU Utilization [%]', estimator='max', b_plot_save=b_plot_save, filename_prefix=filename_prefix)
df_agg.index.name = 'Max ' + collect.df_metrics.loc[metric]['title']
df_agg

,code,experiment_run,client,type_tenants,num_tenants,value
Max CPU Utilization,,,,,,
0,1776418956,1,1,,0,3.89
1,1776418956,1,2,,0,6.29
2,1776418956,2,1,,0,3.42
3,1776418956,2,2,,0,7.55
4,1776420958,1,1,,0,4.60
5,1776420958,1,2,,0,11.47
6,1776420958,2,1,,0,5.81
7,1776420958,2,2,,0,11.64


# Performance Results per Connection

In [23]:
df_performance = collect.get_performance_per_connection()

df_performance_first = df_performance[df_performance['client']==1]
df_performance_second = df_performance[df_performance['client']==2]

df_performance.T

,1776418956-PostgreSQL-BHT-8-1-1-1,1776418956-PostgreSQL-BHT-8-1-2-1,1776418956-PostgreSQL-BHT-8-1-2-2,1776418956-PostgreSQL-BHT-8-2-1-1,1776418956-PostgreSQL-BHT-8-2-2-1,1776418956-PostgreSQL-BHT-8-2-2-2,1776420958-PostgreSQL-BHT-8-1-1-1,1776420958-PostgreSQL-BHT-8-1-2-1,1776420958-PostgreSQL-BHT-8-1-2-2,1776420958-PostgreSQL-BHT-8-2-1-1,1776420958-PostgreSQL-BHT-8-2-2-1,1776420958-PostgreSQL-BHT-8-2-2-2
total_timer_execution,1.57,1.60,1.55,2.35,1.59,1.59,2.91,2.93,2.91,4.24,2.99,2.92
Power@Size [~Q/h],6894.44,6762.32,6966.21,4604.28,6799.26,6798.92,7413.94,7359.71,7429.68,5088.76,7231.64,7401.62
Geo Times [s],1.57,1.60,1.55,2.35,1.59,1.59,2.91,2.93,2.91,4.24,2.99,2.92
configuration,1776418956--,1776418956--,1776418956--,1776418956--,1776418956--,1776418956--,1776420958--,1776420958--,1776420958--,1776420958--,1776420958--,1776420958--
connection,1776418956-PostgreSQL-BHT-8-1-1-1,1776418956-PostgreSQL-BHT-8-1-2-1,1776418956-PostgreSQL-BHT-8-1-2-2,1776418956-PostgreSQL-BHT-8-2-1-1,1776418956-PostgreSQL-BHT-8-2-2-1,1776418956-PostgreSQL-BHT-8-2-2-2,1776420958-PostgreSQL-BHT-8-1-1-1,1776420958-PostgreSQL-BHT-8-1-2-1,1776420958-PostgreSQL-BHT-8-1-2-2,1776420958-PostgreSQL-BHT-8-2-1-1,1776420958-PostgreSQL-BHT-8-2-2-1,1776420958-PostgreSQL-BHT-8-2-2-2
phase,1776418956-PostgreSQL-BHT-8-1-1,1776418956-PostgreSQL-BHT-8-1-2,1776418956-PostgreSQL-BHT-8-1-2,1776418956-PostgreSQL-BHT-8-2-1,1776418956-PostgreSQL-BHT-8-2-2,1776418956-PostgreSQL-BHT-8-2-2,1776420958-PostgreSQL-BHT-8-1-1,1776420958-PostgreSQL-BHT-8-1-2,1776420958-PostgreSQL-BHT-8-1-2,1776420958-PostgreSQL-BHT-8-2-1,1776420958-PostgreSQL-BHT-8-2-2,1776420958-PostgreSQL-BHT-8-2-2
SF,3.00,3.00,3.00,3.00,3.00,3.00,6.00,6.00,6.00,6.00,6.00,6.00
experiment_run,1,1,1,2,2,2,1,1,1,2,2,2
client,1,2,2,1,2,2,1,2,2,1,2,2
time [s],58,60,57,120,59,60,113,110,110,210,119,116


### Add Metadata

In [24]:
df = collect.add_metadata(df_performance)
df.T

combine on columns phase


,1776418956-PostgreSQL-BHT-8-1-1,1776418956-PostgreSQL-BHT-8-1-2,1776418956-PostgreSQL-BHT-8-1-2,1776418956-PostgreSQL-BHT-8-2-1,1776418956-PostgreSQL-BHT-8-2-2,1776418956-PostgreSQL-BHT-8-2-2,1776420958-PostgreSQL-BHT-8-1-1,1776420958-PostgreSQL-BHT-8-1-2,1776420958-PostgreSQL-BHT-8-1-2,1776420958-PostgreSQL-BHT-8-2-1,1776420958-PostgreSQL-BHT-8-2-2,1776420958-PostgreSQL-BHT-8-2-2
total_timer_execution,1.57,1.60,1.55,2.35,1.59,1.59,2.91,2.93,2.91,4.24,2.99,2.92
Power@Size [~Q/h],6894.44,6762.32,6966.21,4604.28,6799.26,6798.92,7413.94,7359.71,7429.68,5088.76,7231.64,7401.62
Geo Times [s],1.57,1.60,1.55,2.35,1.59,1.59,2.91,2.93,2.91,4.24,2.99,2.92
configuration,1776418956--,1776418956--,1776418956--,1776418956--,1776418956--,1776418956--,1776420958--,1776420958--,1776420958--,1776420958--,1776420958--,1776420958--
connection,1776418956-PostgreSQL-BHT-8-1-1-1,1776418956-PostgreSQL-BHT-8-1-2-1,1776418956-PostgreSQL-BHT-8-1-2-2,1776418956-PostgreSQL-BHT-8-2-1-1,1776418956-PostgreSQL-BHT-8-2-2-1,1776418956-PostgreSQL-BHT-8-2-2-2,1776420958-PostgreSQL-BHT-8-1-1-1,1776420958-PostgreSQL-BHT-8-1-2-1,1776420958-PostgreSQL-BHT-8-1-2-2,1776420958-PostgreSQL-BHT-8-2-1-1,1776420958-PostgreSQL-BHT-8-2-2-1,1776420958-PostgreSQL-BHT-8-2-2-2
phase,1776418956-PostgreSQL-BHT-8-1-1,1776418956-PostgreSQL-BHT-8-1-2,1776418956-PostgreSQL-BHT-8-1-2,1776418956-PostgreSQL-BHT-8-2-1,1776418956-PostgreSQL-BHT-8-2-2,1776418956-PostgreSQL-BHT-8-2-2,1776420958-PostgreSQL-BHT-8-1-1,1776420958-PostgreSQL-BHT-8-1-2,1776420958-PostgreSQL-BHT-8-1-2,1776420958-PostgreSQL-BHT-8-2-1,1776420958-PostgreSQL-BHT-8-2-2,1776420958-PostgreSQL-BHT-8-2-2
SF,3.00,3.00,3.00,3.00,3.00,3.00,6.00,6.00,6.00,6.00,6.00,6.00
experiment_run,1,1,1,2,2,2,1,1,1,2,2,2
client,1,2,2,1,2,2,1,2,2,1,2,2
time [s],58,60,57,120,59,60,113,110,110,210,119,116


# Performance Results per Total

In [25]:
df_performance = collect.get_performance_aggregated_per_phase()

df_performance_first = df_performance[df_performance['client']==1]
df_performance_second = df_performance[df_performance['client']==2]

df_performance.dropna(inplace=True)

In [26]:
df_performance.T

,1776418956-PostgreSQL-BHT-8-1-1-1,1776418956-PostgreSQL-BHT-8-1-2-1,1776418956-PostgreSQL-BHT-8-1-2-2,1776418956-PostgreSQL-BHT-8-2-1-1,1776418956-PostgreSQL-BHT-8-2-2-1,1776418956-PostgreSQL-BHT-8-2-2-2,1776420958-PostgreSQL-BHT-8-1-1-1,1776420958-PostgreSQL-BHT-8-1-2-1,1776420958-PostgreSQL-BHT-8-1-2-2,1776420958-PostgreSQL-BHT-8-2-1-1,1776420958-PostgreSQL-BHT-8-2-2-1,1776420958-PostgreSQL-BHT-8-2-2-2
configuration,1776418956--,1776418956--,1776418956--,1776418956--,1776418956--,1776418956--,1776420958--,1776420958--,1776420958--,1776420958--,1776420958--,1776420958--
experiment_run,1,1,1,2,2,2,1,1,1,2,2,2
phase,1776418956-PostgreSQL-BHT-8-1-1,1776418956-PostgreSQL-BHT-8-1-2,1776418956-PostgreSQL-BHT-8-1-2,1776418956-PostgreSQL-BHT-8-2-1,1776418956-PostgreSQL-BHT-8-2-2,1776418956-PostgreSQL-BHT-8-2-2,1776420958-PostgreSQL-BHT-8-1-1,1776420958-PostgreSQL-BHT-8-1-2,1776420958-PostgreSQL-BHT-8-1-2,1776420958-PostgreSQL-BHT-8-2-1,1776420958-PostgreSQL-BHT-8-2-2,1776420958-PostgreSQL-BHT-8-2-2
total_timer_execution,1.57,1.60,1.55,2.35,1.59,1.59,2.91,2.93,2.91,4.24,2.99,2.92
Power@Size [~Q/h],6894.44,6762.32,6966.21,4604.28,6799.26,6798.92,7413.94,7359.71,7429.68,5088.76,7231.64,7401.62
code,1776418956,1776418956,1776418956,1776418956,1776418956,1776418956,1776420958,1776420958,1776420958,1776420958,1776420958,1776420958
pod_count,1,1,1,1,1,1,1,1,1,1,1,1
SF,3.00,3.00,3.00,3.00,3.00,3.00,6.00,6.00,6.00,6.00,6.00,6.00
time [s],58,60,57,120,59,60,113,110,110,210,119,116
client,1,2,2,1,2,2,1,2,2,1,2,2


### With Metadata

In [27]:
df = collect.add_metadata(df_performance)
df.T

combine on columns phase


,1776418956-PostgreSQL-BHT-8-1-1,1776418956-PostgreSQL-BHT-8-1-2,1776418956-PostgreSQL-BHT-8-1-2,1776418956-PostgreSQL-BHT-8-2-1,1776418956-PostgreSQL-BHT-8-2-2,1776418956-PostgreSQL-BHT-8-2-2,1776420958-PostgreSQL-BHT-8-1-1,1776420958-PostgreSQL-BHT-8-1-2,1776420958-PostgreSQL-BHT-8-1-2,1776420958-PostgreSQL-BHT-8-2-1,1776420958-PostgreSQL-BHT-8-2-2,1776420958-PostgreSQL-BHT-8-2-2
configuration,1776418956--,1776418956--,1776418956--,1776418956--,1776418956--,1776418956--,1776420958--,1776420958--,1776420958--,1776420958--,1776420958--,1776420958--
experiment_run,1,1,1,2,2,2,1,1,1,2,2,2
phase,1776418956-PostgreSQL-BHT-8-1-1,1776418956-PostgreSQL-BHT-8-1-2,1776418956-PostgreSQL-BHT-8-1-2,1776418956-PostgreSQL-BHT-8-2-1,1776418956-PostgreSQL-BHT-8-2-2,1776418956-PostgreSQL-BHT-8-2-2,1776420958-PostgreSQL-BHT-8-1-1,1776420958-PostgreSQL-BHT-8-1-2,1776420958-PostgreSQL-BHT-8-1-2,1776420958-PostgreSQL-BHT-8-2-1,1776420958-PostgreSQL-BHT-8-2-2,1776420958-PostgreSQL-BHT-8-2-2
total_timer_execution,1.57,1.60,1.55,2.35,1.59,1.59,2.91,2.93,2.91,4.24,2.99,2.92
Power@Size [~Q/h],6894.44,6762.32,6966.21,4604.28,6799.26,6798.92,7413.94,7359.71,7429.68,5088.76,7231.64,7401.62
code,1776418956,1776418956,1776418956,1776418956,1776418956,1776418956,1776420958,1776420958,1776420958,1776420958,1776420958,1776420958
pod_count,1,1,1,1,1,1,1,1,1,1,1,1
SF,3.00,3.00,3.00,3.00,3.00,3.00,6.00,6.00,6.00,6.00,6.00,6.00
time [s],58,60,57,120,59,60,113,110,110,210,119,116
client,1,2,2,1,2,2,1,2,2,1,2,2


In [28]:
df[['phase', 'code', 'configuration', 'experiment_run', 'client', 'type_tenants', 'num_tenants', 'vol_tenants']]

,phase,code,configuration,experiment_run,client,type_tenants,num_tenants,vol_tenants
1776418956-PostgreSQL-BHT-8-1-1,1776418956-PostgreSQL-BHT-8-1-1,1776418956,1776418956--,1,1,,0,False
1776418956-PostgreSQL-BHT-8-1-2,1776418956-PostgreSQL-BHT-8-1-2,1776418956,1776418956--,1,2,,0,False
1776418956-PostgreSQL-BHT-8-1-2,1776418956-PostgreSQL-BHT-8-1-2,1776418956,1776418956--,1,2,,0,False
1776418956-PostgreSQL-BHT-8-2-1,1776418956-PostgreSQL-BHT-8-2-1,1776418956,1776418956--,2,1,,0,False
1776418956-PostgreSQL-BHT-8-2-2,1776418956-PostgreSQL-BHT-8-2-2,1776418956,1776418956--,2,2,,0,False
1776418956-PostgreSQL-BHT-8-2-2,1776418956-PostgreSQL-BHT-8-2-2,1776418956,1776418956--,2,2,,0,False
1776420958-PostgreSQL-BHT-8-1-1,1776420958-PostgreSQL-BHT-8-1-1,1776420958,1776420958--,1,1,,0,False
1776420958-PostgreSQL-BHT-8-1-2,1776420958-PostgreSQL-BHT-8-1-2,1776420958,1776420958--,1,2,,0,False
1776420958-PostgreSQL-BHT-8-1-2,1776420958-PostgreSQL-BHT-8-1-2,1776420958,1776420958--,1,2,,0,False
1776420958-PostgreSQL-BHT-8-2-1,1776420958-PostgreSQL-BHT-8-2-1,1776420958,1776420958--,2,1,,0,False


In [29]:
collectors.get_non_constant(df).T

,1776418956-PostgreSQL-BHT-8-1-1,1776418956-PostgreSQL-BHT-8-1-2,1776418956-PostgreSQL-BHT-8-1-2,1776418956-PostgreSQL-BHT-8-2-1,1776418956-PostgreSQL-BHT-8-2-2,1776418956-PostgreSQL-BHT-8-2-2,1776420958-PostgreSQL-BHT-8-1-1,1776420958-PostgreSQL-BHT-8-1-2,1776420958-PostgreSQL-BHT-8-1-2,1776420958-PostgreSQL-BHT-8-2-1,1776420958-PostgreSQL-BHT-8-2-2,1776420958-PostgreSQL-BHT-8-2-2
configuration,1776418956--,1776418956--,1776418956--,1776418956--,1776418956--,1776418956--,1776420958--,1776420958--,1776420958--,1776420958--,1776420958--,1776420958--
experiment_run,1,1,1,2,2,2,1,1,1,2,2,2
phase,1776418956-PostgreSQL-BHT-8-1-1,1776418956-PostgreSQL-BHT-8-1-2,1776418956-PostgreSQL-BHT-8-1-2,1776418956-PostgreSQL-BHT-8-2-1,1776418956-PostgreSQL-BHT-8-2-2,1776418956-PostgreSQL-BHT-8-2-2,1776420958-PostgreSQL-BHT-8-1-1,1776420958-PostgreSQL-BHT-8-1-2,1776420958-PostgreSQL-BHT-8-1-2,1776420958-PostgreSQL-BHT-8-2-1,1776420958-PostgreSQL-BHT-8-2-2,1776420958-PostgreSQL-BHT-8-2-2
total_timer_execution,1.57,1.60,1.55,2.35,1.59,1.59,2.91,2.93,2.91,4.24,2.99,2.92
Power@Size [~Q/h],6894.44,6762.32,6966.21,4604.28,6799.26,6798.92,7413.94,7359.71,7429.68,5088.76,7231.64,7401.62
code,1776418956,1776418956,1776418956,1776418956,1776418956,1776418956,1776420958,1776420958,1776420958,1776420958,1776420958,1776420958
SF,3.00,3.00,3.00,3.00,3.00,3.00,6.00,6.00,6.00,6.00,6.00,6.00
time [s],58,60,57,120,59,60,113,110,110,210,119,116
client,1,2,2,1,2,2,1,2,2,1,2,2
Throughput@Size,4096.55,3960.00,4168.42,1980.00,4027.12,3960.00,4205.31,4320.00,4320.00,2262.86,3993.28,4096.55


# Performance at Ingestion

In [30]:
df_performance = collect.get_connections()
#df_performance = collect.get_loading_time_max_all()

df_performance_first = df_performance[df_performance['client']==1]
df_performance_second = df_performance[df_performance['client']==2]

df_performance.T

,1776418956-PostgreSQL-BHT-8-1-1,1776418956-PostgreSQL-BHT-8-1-2,1776418956-PostgreSQL-BHT-8-2-1,1776418956-PostgreSQL-BHT-8-2-2,1776420958-PostgreSQL-BHT-8-1-1,1776420958-PostgreSQL-BHT-8-1-2,1776420958-PostgreSQL-BHT-8-2-1,1776420958-PostgreSQL-BHT-8-2-2
code,1776418956,1776418956,1776418956,1776418956,1776420958,1776420958,1776420958,1776420958
connection,PostgreSQL-BHT-8-1-1-1,PostgreSQL-BHT-8-1-2-2,PostgreSQL-BHT-8-2-1-1,PostgreSQL-BHT-8-2-2-2,PostgreSQL-BHT-8-1-1-1,PostgreSQL-BHT-8-1-2-2,PostgreSQL-BHT-8-2-1-1,PostgreSQL-BHT-8-2-2-2
configuration,,,,,,,,
phase,1776418956-PostgreSQL-BHT-8-1-1,1776418956-PostgreSQL-BHT-8-1-2,1776418956-PostgreSQL-BHT-8-2-1,1776418956-PostgreSQL-BHT-8-2-2,1776420958-PostgreSQL-BHT-8-1-1,1776420958-PostgreSQL-BHT-8-1-2,1776420958-PostgreSQL-BHT-8-2-1,1776420958-PostgreSQL-BHT-8-2-2
experiment_run,1,1,2,2,1,1,2,2
client,1,2,1,2,1,2,1,2
dockerimage,postgres:18.3,postgres:18.3,postgres:18.3,postgres:18.3,postgres:18.3,postgres:18.3,postgres:18.3,postgres:18.3
time_load,455.00,455.00,455.00,455.00,1256.00,1256.00,1256.00,1256.00
time_ingest,141.00,141.00,141.00,141.00,350.00,350.00,350.00,350.00
time_check,291.00,291.00,291.00,291.00,843.00,843.00,843.00,843.00


# Hardware Monitoring for Loading Phase

In [31]:
df_performance = collect.get_monitoring_aggregated_per_phase("loading")
df_performance = collect.add_metadata(df_performance)
#df_performance_first = df_performance[df_performance['client']==1]
#df_performance_second = df_performance[df_performance['client']==2]

df_performance.T

combine on index and column 'phase'


phase,1776418956-PostgreSQL-BHT-8-1-1,1776418956-PostgreSQL-BHT-8-1-2,1776420958-PostgreSQL-BHT-8-1-1,1776420958-PostgreSQL-BHT-8-1-2
Memory Usage [MiB],7122.96,7122.96,9493.28,9493.28
Memory Usage Cached [MiB],9772.04,9772.04,15577.73,15577.73
CPU Utilization,1.08,1.08,0.95,0.95
CPU Throttle,0.00,0.00,0.00,0.00
CPU Throttled Time [s],0,0,0,0
CPU Utilization Time [s],303.26,303.26,819.77,819.77
CPU User Time [s],236.03,236.03,686.43,686.43
CPU System Time [s],67.23,67.23,133.34,133.34
Network Rx Total [MiB],3319.50,3319.50,6675.75,6675.75
Network Tx Total [MiB],2624.35,2624.35,6061.46,6061.46


In [32]:
#plot_bars(df_performance, y='CPU Utilization Time [s]', title='CPU [CPUs]', estimator='max', b_plot_save=b_plot_save, filename_prefix=filename_prefix)

In [33]:
#plot_bars(df_performance, y='Memory Usage [MiB]', title='Memory Usage [MiB]', estimator='max', b_plot_save=b_plot_save, filename_prefix=filename_prefix)

In [34]:
#plot_bars(df_performance, y='Memory Usage Cached [MiB]', title='Memory Usage Cached [MiB]', estimator='max', b_plot_save=b_plot_save, filename_prefix=filename_prefix)

# Efficiency

In [35]:
client = 1

df_performance_monitoring = collect.get_monitoring_aggregated_per_phase(type="stream")
#df_performance_monitoring = df_performance_monitoring[df_performance_monitoring['client'] == client]
df_performance = collect.get_performance_aggregated_per_phase()
#df_performance = df_performance[df_performance['client'] == client]
merged_df = pd.merge(df_performance, df_performance_monitoring, left_index=True, right_index=True, how='left')
#merged_df = pd.merge(df_performance, df_performance_monitoring, on=['phase'], how='inner')
#merged_df['I_Lat'] = 1./merged_df['E_Lat']
merged_df = merged_df[merged_df['client'] == client]
#merged_df['E_Tpx'] = merged_df['Goodput (requests/second)'] / merged_df['CPU Utilization Time [s]'] * 600.
#merged_df['E_Lat'] = 1./np.sqrt(merged_df['Latency Distribution.Average Latency (microseconds)']*merged_df['CPU Utilization Time [s]']/1E6)
#merged_df['E_RAM'] = (merged_df['Goodput (requests/second)']) / merged_df['Memory Usage [MiB]']
merged_df.T

,1776418956-PostgreSQL-BHT-8-1-1-1,1776418956-PostgreSQL-BHT-8-2-1-1,1776420958-PostgreSQL-BHT-8-1-1-1,1776420958-PostgreSQL-BHT-8-2-1-1
configuration,1776418956--,1776418956--,1776420958--,1776420958--
experiment_run,1,2,1,2
phase,1776418956-PostgreSQL-BHT-8-1-1,1776418956-PostgreSQL-BHT-8-2-1,1776420958-PostgreSQL-BHT-8-1-1,1776420958-PostgreSQL-BHT-8-2-1
total_timer_execution,1.57,2.35,2.91,4.24
Power@Size [~Q/h],6894.44,4604.28,7413.94,5088.76
code,1776418956,1776418956,1776420958,1776420958
pod_count,1,1,1,1
SF,3.00,3.00,6.00,6.00
time [s],58,120,113,210
client,1,1,1,1


In [36]:
df = collect.add_metadata(merged_df)
df.T

combine on columns phase


,1776418956-PostgreSQL-BHT-8-1-1,1776418956-PostgreSQL-BHT-8-1-2,1776418956-PostgreSQL-BHT-8-2-1,1776418956-PostgreSQL-BHT-8-2-2,1776420958-PostgreSQL-BHT-8-1-1,1776420958-PostgreSQL-BHT-8-1-2,1776420958-PostgreSQL-BHT-8-2-1,1776420958-PostgreSQL-BHT-8-2-2
configuration,1776418956--,,1776418956--,,1776420958--,,1776420958--,
experiment_run,1.00,1,2.00,2,1.00,1,2.00,2
phase,1776418956-PostgreSQL-BHT-8-1-1,1776418956-PostgreSQL-BHT-8-1-2,1776418956-PostgreSQL-BHT-8-2-1,1776418956-PostgreSQL-BHT-8-2-2,1776420958-PostgreSQL-BHT-8-1-1,1776420958-PostgreSQL-BHT-8-1-2,1776420958-PostgreSQL-BHT-8-2-1,1776420958-PostgreSQL-BHT-8-2-2
total_timer_execution,1.57,NaN,2.35,NaN,2.91,NaN,4.24,NaN
Power@Size [~Q/h],6894.44,NaN,4604.28,NaN,7413.94,NaN,5088.76,NaN
code,1776418956,1776418956,1776418956,1776418956,1776420958,1776420958,1776420958,1776420958
pod_count,1.00,NaN,1.00,NaN,1.00,NaN,1.00,NaN
SF,3.00,NaN,3.00,NaN,6.00,NaN,6.00,NaN
time [s],58.00,NaN,120.00,NaN,113.00,NaN,210.00,NaN
client,1.00,2,1.00,2,1.00,2,1.00,2


In [37]:
df[['phase', 'code', 'configuration', 'experiment_run', 'client', 'type_tenants', 'num_tenants', 'vol_tenants']]

,phase,code,configuration,experiment_run,client,type_tenants,num_tenants,vol_tenants
1776418956-PostgreSQL-BHT-8-1-1,1776418956-PostgreSQL-BHT-8-1-1,1776418956,1776418956--,1.00,1.00,,0,False
1776418956-PostgreSQL-BHT-8-1-2,1776418956-PostgreSQL-BHT-8-1-2,1776418956,,1,2,,0,False
1776418956-PostgreSQL-BHT-8-2-1,1776418956-PostgreSQL-BHT-8-2-1,1776418956,1776418956--,2.00,1.00,,0,False
1776418956-PostgreSQL-BHT-8-2-2,1776418956-PostgreSQL-BHT-8-2-2,1776418956,,2,2,,0,False
1776420958-PostgreSQL-BHT-8-1-1,1776420958-PostgreSQL-BHT-8-1-1,1776420958,1776420958--,1.00,1.00,,0,False
1776420958-PostgreSQL-BHT-8-1-2,1776420958-PostgreSQL-BHT-8-1-2,1776420958,,1,2,,0,False
1776420958-PostgreSQL-BHT-8-2-1,1776420958-PostgreSQL-BHT-8-2-1,1776420958,1776420958--,2.00,1.00,,0,False
1776420958-PostgreSQL-BHT-8-2-2,1776420958-PostgreSQL-BHT-8-2-2,1776420958,,2,2,,0,False
